# 4. Pilot Comparison

This notebook launches matched pilot runs for the baseline and AttnRes models and then compares them with the helper scripts.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/AtinChing/AttnResGPT-mini.git'
REPO_NAME = 'AttnResGPT-mini'

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

candidates = [Path(f'/content/{REPO_NAME}'), Path(f'/content/drive/MyDrive/{REPO_NAME}'), Path.cwd()]
repo_root = next((p for p in candidates if (p / 'requirements.txt').exists() and (p / 'src').exists()), None)

if repo_root is None:
    target = Path(f'/content/{REPO_NAME}')
    print(f'Cloning {REPO_URL} into {target} ...')
    subprocess.run(['git', 'clone', REPO_URL, str(target)], check=True)
    repo_root = target
else:
    print(f'Using existing repo at {repo_root}')

%cd {repo_root}
!pip -q install -r requirements.txt

In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name:', torch.cuda.get_device_name(0))
    print('bf16_supported:', torch.cuda.is_bf16_supported())

In [ ]:
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb4_pilot_baseline model.architecture=baseline
!python -m src.train --config configs/pilot_t4.yaml --overrides experiment.name=nb4_pilot_attnres model.architecture=attnres model.attnres.enabled=true

In [ ]:
from pathlib import Path

baseline_run = sorted(Path('runs').glob('nb4_pilot_baseline_*'))[-1]
attnres_run = sorted(Path('runs').glob('nb4_pilot_attnres_*'))[-1]
!python scripts/compare_runs.py --baseline-run {baseline_run} --attnres-run {attnres_run}
!python scripts/plot_metrics.py --run-dirs {baseline_run} {attnres_run} --output-dir plots/nb4_pilot